# 04. Карты для презентации (дизайн-система maptools)

6 карт в едином макете (2000×1400 px, подложки Esri, линейка, стрелка севера, легенда справа):

1. Четыре версии границ; 2. Возраст застройки; 3. Объекты культурного наследия; 4. Сервисы и зоны пешей доступности; 5. Зелень и промышленность; 6. Плотность жилой застройки (гексагональная сетка); 7. Пространственная модель (зоны, узлы, каркасы).

PNG в `notebooks/maps_out/` (в git не входят), вставляются в `presentations/` без подгонки.

In [1]:
# Настройка: maptools в sys.path, загрузка слоёв, охват карт
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "scripts"))

import maptools  # noqa: E402

PROC = PROJECT / "data/processed"
CACHE = PROJECT / "data/raw/cache"
OUT = PROJECT / "notebooks/maps_out"
OUT.mkdir(exist_ok=True)
CRS_M = 32636

versions = gpd.read_file(PROC / "boundary_versions.geojson").set_index("version")
V1, V2, V3, V4 = (versions.loc[v, "geometry"] for v in ["v1", "v2", "v3", "v4"])
building = gpd.read_file(PROC / "building.geojson")
# классификация застройки (как в ноутбуке 03)
LIVING = ["apartments", "residential", "dormitory", "detached", "house", "semidetached_house", "yes"]
building["kind"] = "прочая"
building.loc[building.building.isin(["industrial", "warehouse"]), "kind"] = "промышленно-складская"
building.loc[building.building.isin(LIVING), "kind"] = "жилая"

green = gpd.read_file(PROC / "green.geojson")
services = gpd.read_file(PROC / "services.geojson")
water = gpd.read_file(PROC / "water.geojson")
roads = gpd.read_file(PROC / "roads.geojson")
house_age = gpd.read_file(PROC / "house_age.geojson")
okn = gpd.read_file(PROC / "okn.geojson")

# осевые рек из кэша Overpass (для надёжной отрисовки воды поверх неполного полигона Невы)
import json
from shapely.geometry import LineString
wp = json.load(open(CACHE / "overpass_water_bbox.json", encoding="utf-8"))
axes_neva = [LineString([(g["lon"], g["lat"]) for g in el["geometry"]])
             for el in wp["elements"] if el.get("tags", {}).get("name") == "Нева"]
axes_okhta = [LineString([(g["lon"], g["lat"]) for g in el["geometry"]])
              for el in wp["elements"] if el.get("tags", {}).get("name") == "Охта"]

WATER_COLOR = "#85add3"

WATER_ALL = water.to_crs(CRS_M).union_all().union(
    gpd.GeoSeries(axes_neva, crs=4326).to_crs(CRS_M).buffer(260).union_all()).union(
    gpd.GeoSeries(axes_okhta, crs=4326).to_crs(CRS_M).buffer(45).union_all()).buffer(0)
LAND_V3 = gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M).iloc[0].difference(WATER_ALL).buffer(0)
LAND_V2 = gpd.GeoSeries([V2], crs=4326).to_crs(CRS_M).iloc[0].difference(WATER_ALL).buffer(0)

def draw_water(ax):
    """Вода: полигональный слой + полосы вдоль осевых (полигон Невы из OSM неполный)."""
    water.to_crs(CRS_M).plot(ax=ax, facecolor=WATER_COLOR, edgecolor="none", alpha=0.9, zorder=1)
    gpd.GeoSeries(axes_neva, crs=4326).to_crs(CRS_M).buffer(240).plot(
        ax=ax, facecolor=WATER_COLOR, edgecolor="none", alpha=0.9, zorder=1)
    gpd.GeoSeries(axes_okhta, crs=4326).to_crs(CRS_M).buffer(45).plot(
        ax=ax, facecolor=WATER_COLOR, edgecolor="none", alpha=0.9, zorder=1)

def snap_to_land(gdf):
    """Точки, попавшие в воду (привязка грубая), сдвигаем к ближайшей точке берега."""
    g = gdf.to_crs(CRS_M).copy()
    inside = g.geometry.within(WATER_ALL)
    for i in g.index[inside]:
        p = g.at[i, "geometry"]
        g.at[i, "geometry"] = p.intersection(WATER_ALL).buffer(0) if False else _nearest_land(p)
    return g.to_crs(4326)

def _nearest_land(p):
    from shapely.ops import nearest_points
    q = nearest_points(LAND_V3, p)[0]
    return q

# охват в метрах UTM 36N с запасом 300 м
b = gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M)
minx, miny, maxx, maxy = b.total_bounds
EXTENT = (minx - 300, miny - 300, maxx + 300, maxy + 300)
V2_m = gpd.GeoSeries([V2], crs=4326).to_crs(CRS_M)
print("охват карт, м:", [round(v) for v in EXTENT])

охват карт, м: [354073, 6644527, 357751, 6650059]


In [2]:
# Карта 1: территория стратегии — три уровня. Границы по береговой линии, не по середине воды.
# Историческое ядро — по историческим сведениям: кварталы дореволюционной застройки (HOITH)
# и устье Охты с Охтинским мысом (Ниеншанц, Охтинская слобода у верфи Петра I).
# Дореволюционная застройка Малой Охты — деревянная, вдоль Октябрьской набережной;
# рост района после постройки Большеохтинского моста (1909–1911).
v3_land = gpd.GeoSeries([LAND_V3], crs=CRS_M)
v2_land = gpd.GeoSeries([LAND_V2], crs=CRS_M)

dorev = house_age[(house_age.year < 1917)].to_crs(CRS_M)
core_buf = dorev.buffer(250).union_all()
from shapely.ops import nearest_points
mouth = nearest_points(
    gpd.GeoSeries(axes_neva, crs=4326).to_crs(CRS_M).union_all(),
    gpd.GeoSeries(axes_okhta, crs=4326).to_crs(CRS_M).union_all())[0]
mouth_buf = gpd.GeoSeries([mouth], crs=CRS_M).buffer(500).union_all()
V2_full = gpd.GeoSeries([V2], crs=4326).to_crs(CRS_M).iloc[0]
core = core_buf.union(mouth_buf).intersection(V2_full).intersection(LAND_V3).buffer(0)
core_g = gpd.GeoSeries([core], crs=CRS_M)

fig, ax = maptools.map_figure(full=True)
h = []
v3_land.plot(ax=ax, facecolor="#dbe7f0", edgecolor="#4a7aa5", linewidth=1.4, alpha=0.9, zorder=1)
h.append(Patch(facecolor="#dbe7f0", edgecolor="#4a7aa5", label="Два охтинских муниципальных округа"))
v2_land.boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=2.4, zorder=4)
h.append(Line2D([0], [0], color="#1f4e79", lw=2.4, label="МО Малая Охта — территория стратегии"))
core_g.plot(ax=ax, facecolor="#e9b96e", edgecolor="#8c2d2d", linewidth=2.2, alpha=0.9, zorder=5)
h.append(Patch(facecolor="#e9b96e", edgecolor="#8c2d2d", label="Историческое ядро: дореволюционные кварталы и берег у мыса"))
draw_water(ax)
h.append(Patch(facecolor=WATER_COLOR, alpha=0.9, label="Вода"))

for text, geom, dx, dy, color in [
    ("МО Малая Охта", V2, 0, -300, "#1f4e79"),
    ("МО Большая Охта", versions.loc["v3", "geometry"].difference(V2), 0, 600, "#4a7aa5"),
]:
    p = gpd.GeoSeries([geom], crs=4326).to_crs(CRS_M).geometry.iloc[0].representative_point()
    ax.text(p.x + dx, p.y + dy, text, ha="center", va="center", fontsize=11, fontweight="bold",
            color=color, path_effects=maptools.HALO, zorder=8)
p1 = core_g.geometry.iloc[0].representative_point()
ax.text(p1.x - 300, p1.y + 1100, "Охтинский мыс", ha="center", va="center", fontsize=10, fontweight="bold",
        color="#8c2d2d", path_effects=maptools.HALO, zorder=8)

maptools.finish(fig, ax, EXTENT, h, title="Территория", crs=CRS_M, basemap_style="gray", basemap_alpha=0.95)
maptools.save_map(fig, OUT / "01_boundaries.png")
print("1 готова")


1 готова


In [3]:
# Карта 6: плотность жилой застройки — сколько жилых домов в каждой ячейке
# Сетка: шестиугольники радиусом 250 м; цвет и легенда — одна классификация.
import numpy as np
from shapely.geometry import Polygon

R = 250  # радиус ячейки, м (площадь ~16,2 га)
living = building[building.kind == "жилая"]
liv_m = living.to_crs(CRS_M)

x0, y0, x1, y1 = EXTENT
dx = R * np.sqrt(3); dy = R * 1.5
cells = []
for i, yc in enumerate(np.arange(y0, y1 + dy, dy)):
    for xc in np.arange(x0, x1 + dx, dx) + (dx / 2 if i % 2 else 0):
        pts = [(xc + R * np.cos(a), yc + R * np.sin(a)) for a in np.linspace(0, 2 * np.pi, 7)[:-1]]
        cells.append(Polygon(pts))
grid = gpd.GeoDataFrame(geometry=cells, crs=CRS_M)
grid = grid[grid.intersects(gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M).iloc[0])]

v3_m = gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M).iloc[0].buffer(0)
liv_v3_m = liv_m[liv_m.within(gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M).iloc[0])]
liv_union = liv_v3_m.geometry.union_all()  # только жилые дома на территории стратегии
grid["footprint"] = grid.geometry.intersection(liv_union).area / 1e4  # га жилых домов в ячейке

CLASSES = [("мало: до 0,5 га", 0.0, 0.5, "#fdd49e"), ("средне: 0,5–2 га", 0.5, 2.0, "#fc8d59"),
           ("много: более 2 га", 2.0, 1e9, "#d73027")]
grid["cls"] = np.select(
    [grid.footprint.between(lo, hi, inclusive="right") for _, lo, hi, _ in CLASSES],
    range(len(CLASSES)), default=-1)

fig, ax = maptools.map_figure(full=True)
h = []
for k, (lab, _, _, color) in enumerate(CLASSES):
    sub = grid[grid.cls == k]
    if len(sub):
        sub.plot(ax=ax, facecolor=color, edgecolor="white", linewidth=0.3, zorder=3)
    h.append(Patch(facecolor=color, edgecolor="white", label=f"Жилые дома в ячейке: {lab} ({len(sub)})"))
draw_water(ax)
h.append(Patch(facecolor=WATER_COLOR, alpha=0.9, label="Вода"))
gpd.GeoSeries([LAND_V2], crs=CRS_M).boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.6, zorder=5)
h.append(Line2D([0], [0], color="#1f4e79", lw=1.6, label="Граница МО Малая Охта"))

maptools.finish(fig, ax, EXTENT, h, title="Плотность жилой застройки", crs=CRS_M, basemap_style="gray", basemap_alpha=0.95)
maptools.save_map(fig, OUT / "06_density.png")
print("6 готова; ячеек с застройкой:", int((grid.cls >= 0).sum()))


6 готова; ячеек с застройкой: 74


In [4]:
# Карта 7: пространственная модель — зоны, узлы, направления
# Зоны из данных: ядро = v1; берег = 300 м от оси Охты в v3; восток = за Заневским проспектом;
# остальное — жилые кварталы. Линии направлений — рабочая схема, не трассировка.
from shapely.geometry import box as shp_box

v3_m = gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M).iloc[0].buffer(0)
v1_m = gpd.GeoSeries([V1], crs=4326).to_crs(CRS_M).iloc[0].buffer(0)

okhta_axis = gpd.GeoSeries(axes_okhta, crs=4326).to_crs(CRS_M).union_all()
z2_raw = okhta_axis.buffer(300).intersection(v3_m).difference(v1_m).buffer(0)

zanev = roads[roads.name == "Заневский проспект"].to_crs(CRS_M)
x_zanev = zanev.geometry.union_all().centroid.x
z4_raw = v3_m.intersection(shp_box(x_zanev, v3_m.bounds[1], v3_m.bounds[2] + 10, v3_m.bounds[3] + 10)).buffer(0)
z4 = z4_raw.difference(v1_m).difference(z2_raw).buffer(0)
z2 = z2_raw.difference(z4).buffer(0)
z1 = v1_m
z3 = v3_m.difference(v1_m).difference(z2).difference(z4).buffer(0)

def node_xy(gdf_series):
    p = gdf_series.to_crs(CRS_M).geometry.iloc[0]
    if p.geom_type != "Point":
        p = p.centroid
    return p.x, p.y

if "stations" not in dir():
    stations = gpd.read_file(PROC / "stations.geojson")
st_metro = stations[stations.name.isin(["Новочеркасская", "Ладожская"])]
utkina = okn[okn.name.str.contains("Уткина", na=False)]
niens = okn[okn.name.str.contains("Ниеншанц", na=False)]
nodes = {}
nodes["Охтинский мыс"] = node_xy(snap_to_land(niens))
nodes["Уткина дача"] = node_xy(snap_to_land(utkina))
for _, r in st_metro.iterrows():
    nodes[r["name"]] = node_xy(gpd.GeoSeries([r.geometry], crs=4326))

fig, ax = maptools.map_figure(full=True)
h = []
ZONES = [(z1, "#e9b96e", "Историческое ядро — флагман"),
         (z2, "#c98d3a", "Берег Охты — набережная и новые функции"),
         (z3, "#dcd8d0", "Жилые кварталы — наполнение середины"),
         (z4, "#a9c79a", "Восточный резерв — зелёное развитие, 3-я очередь")]
for geom, color, lab in ZONES:
    if geom.area > 100:
        gpd.GeoSeries([geom], crs=CRS_M).plot(ax=ax, facecolor=color, edgecolor="none", alpha=0.75, zorder=2)
        h.append(Patch(facecolor=color, alpha=0.75, edgecolor="none", label=lab))

draw_water(ax)
h.append(Patch(facecolor=WATER_COLOR, alpha=0.9, label="Вода"))

ax_m = gpd.GeoSeries(axes_neva + axes_okhta, crs=4326).to_crs(CRS_M)
ax_m[ax_m.intersects(v3_m)].plot(ax=ax, color="#2b6cb0", linewidth=2.6, zorder=6)
h.append(Line2D([0], [0], color="#2b6cb0", lw=2.6, label="Водная общественная ось"))

from shapely.geometry import LineString
if len(utkina) and len(niens):
    hx, hy = nodes["Охтинский мыс"]; ux, uy = nodes["Уткина дача"]
    ax.plot([hx, ux], [hy, uy], color="#8c2d2d", lw=2.0, ls="--", zorder=6)
h.append(Line2D([0], [0], color="#8c2d2d", lw=2.0, ls="--", label="Историческая связь: мыс — Уткина дача"))

nx_, ny_ = nodes["Новочеркасская"]
lx, ly = nodes["Ладожская"]
gx1 = v3_m.bounds[2] - 150
ax.plot([nx_, lx, gx1], [ny_, ly, ly - 250], color="#4f772d", lw=2.0, ls="--", zorder=6)
h.append(Line2D([0], [0], color="#4f772d", lw=2.0, ls="--", label="Зелёно-пешеходное направление к паркам"))

from matplotlib.patches import Circle as MplCircle
for nm in ("Новочеркасская", "Ладожская"):
    px_, py_ = nodes[nm]
    ax.add_patch(MplCircle((px_, py_), 400, fill=False, edgecolor="#52514e", lw=1.4, ls=":", zorder=6))
h.append(Line2D([0], [0], color="#52514e", lw=1.4, ls=":", label="Всё нужное в 400 м от метро"))

for nm, (px_, py_) in nodes.items():
    ax.plot(px_, py_, marker="*", color="#0b0b0b", markersize=16, zorder=8, path_effects=maptools.HALO)
    ax.annotate(nm, (px_, py_), xytext=(px_ + 60, py_ + 90), fontsize=9, fontweight="bold",
                color="#0b0b0b", path_effects=maptools.HALO, zorder=9)
h.append(Line2D([0], [0], marker="*", color="none", markerfacecolor="#0b0b0b", markersize=13, label="Точки развития"))

gpd.GeoSeries([LAND_V2], crs=CRS_M).boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.6, zorder=5)
h.append(Line2D([0], [0], color="#1f4e79", lw=1.6, label="Граница МО Малая Охта"))

maptools.finish(fig, ax, EXTENT, h, title="Модель территории", crs=CRS_M, basemap_style="gray", basemap_alpha=0.9)
maptools.save_map(fig, OUT / "07_spatial_model.png")
print("7 готова; x Заневского:", round(x_zanev))


7 готова; x Заневского: 355904


In [5]:
# Карта 5: что занимает землю — зелень, промзоны, гаражи, жилые дома
fig, ax = maptools.map_figure(full=True)
h = []
draw_water(ax)
green[green.kind.isin(["park", "garden"])].to_crs(CRS_M).plot(
    ax=ax, facecolor="#7fb069", edgecolor="none", alpha=0.8, zorder=2)
h.append(Patch(facecolor="#7fb069", alpha=0.8, label="Парки и сады"))
green[green.kind.isin(["forest", "wood", "grass", "grassland", "meadow", "scrub"])].to_crs(CRS_M).plot(
    ax=ax, facecolor="#c5d9a9", edgecolor="none", alpha=0.75, zorder=2)
h.append(Patch(facecolor="#c5d9a9", alpha=0.75, label="Лес и травяная зелень"))
green[green.kind == "industrial"].to_crs(CRS_M).plot(
    ax=ax, facecolor="#8a6d3b", edgecolor="none", alpha=0.75, zorder=3)
h.append(Patch(facecolor="#8a6d3b", alpha=0.75, label="Промзоны (в OSM недоучтены)"))
building[building.building == "industrial"].to_crs(CRS_M).plot(
    ax=ax, facecolor="#6b4f2a", edgecolor="none", alpha=0.9, zorder=4)
h.append(Patch(facecolor="#6b4f2a", alpha=0.9, label="Промышленные здания"))
building[building.building == "garages"].to_crs(CRS_M).plot(
    ax=ax, facecolor="#b34a4a", edgecolor="none", alpha=0.9, zorder=4)
h.append(Patch(facecolor="#b34a4a", alpha=0.9, label="Гаражные массивы"))
building[building.kind == "жилая"].to_crs(CRS_M).plot(
    ax=ax, facecolor="#d8d4cc", edgecolor="#b5b0a6", linewidth=0.25, alpha=0.9, zorder=3)
h.append(Patch(facecolor="#d8d4cc", edgecolor="#b5b0a6", label="Жилые дома"))
h.append(Patch(facecolor=WATER_COLOR, alpha=0.9, label="Вода"))
gpd.GeoSeries([LAND_V2], crs=CRS_M).boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.6, zorder=5)
h.append(Line2D([0], [0], color="#1f4e79", lw=1.6, label="Граница МО Малая Охта"))

maptools.finish(fig, ax, EXTENT, h, title="Землепользование", crs=CRS_M, basemap_style="gray", basemap_alpha=0.95)
maptools.save_map(fig, OUT / "05_landuse.png")
print("5 готова")


5 готова


In [6]:
# Карта 4: сколько жилых домов доходит пешком до школы и детсада
fig, ax = maptools.map_figure(full=True)
h = []
schools = services[services.amenity == "school"]
kinder = services[services.amenity == "kindergarten"]

z_s = gpd.read_file(PROC / "coverage_schools.geojson").to_crs(CRS_M)
z_s.plot(ax=ax, facecolor="#6b5ca5", alpha=0.4, edgecolor="none", zorder=2)
h.append(Patch(facecolor="#6b5ca5", alpha=0.45, label="До школы меньше 500 м пешком"))
z_k = gpd.read_file(PROC / "coverage_kindergartens.geojson").to_crs(CRS_M)
z_k.plot(ax=ax, facecolor="#7fb069", alpha=0.45, edgecolor="none", zorder=3)
h.append(Patch(facecolor="#7fb069", alpha=0.5, label="До детсада меньше 300 м пешком"))

living = building[building.kind == "жилая"]
living.to_crs(CRS_M).plot(ax=ax, facecolor="#d8d4cc", edgecolor="#b5b0a6", linewidth=0.25, zorder=3)
h.append(Patch(facecolor="#d8d4cc", edgecolor="#b5b0a6", label="Жилые дома"))
kinder = snap_to_land(kinder)
kinder.to_crs(CRS_M).plot(ax=ax, color="#4f772d", markersize=22, edgecolor="white", linewidth=0.5, zorder=6)
h.append(Line2D([0], [0], marker="o", color="none", markerfacecolor="#4f772d", markeredgecolor="white",
                markersize=7, label=f"Детсады ({len(kinder)})"))
schools = snap_to_land(schools)
schools.to_crs(CRS_M).plot(ax=ax, color="#3a2d6b", markersize=34, edgecolor="white", linewidth=0.6, zorder=7)
h.append(Line2D([0], [0], marker="o", color="none", markerfacecolor="#1f4e79", markeredgecolor="white",
                markersize=9, label=f"Школы ({len(schools)})"))
draw_water(ax)
h.append(Patch(facecolor=WATER_COLOR, alpha=0.9, label="Вода"))
gpd.GeoSeries([LAND_V2], crs=CRS_M).boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.6, zorder=5)
h.append(Line2D([0], [0], color="#1f4e79", lw=1.6, label="Граница МО Малая Охта"))

maptools.finish(fig, ax, EXTENT, h, title="Доступность пешком", crs=CRS_M, basemap_style="gray", basemap_alpha=0.95)
maptools.save_map(fig, OUT / "04_services.png")
print("4 готова (сетевые зоны)")


4 готова (сетевые зоны)


In [7]:
# Карта 3: объекты культурного наследия + Ниеншанц
fig, ax = maptools.map_figure(full=True)
h = []
draw_water(ax)
h.append(Patch(facecolor=WATER_COLOR, alpha=0.9, label="Вода"))
green[green.kind.isin(["park", "forest", "wood"])].to_crs(CRS_M).plot(
    ax=ax, facecolor="#a9c79a", edgecolor="none", alpha=0.55, zorder=2)
h.append(Patch(facecolor="#a9c79a", alpha=0.55, label="Парки и зелень"))

okn_d = snap_to_land(okn)
fed = okn_d[okn_d.cat == "Федерального значения"]
reg = okn_d[okn_d.cat == "Регионального значения"]
reg.to_crs(CRS_M).plot(ax=ax, color="#c98d3a", markersize=26, edgecolor="white", linewidth=0.6, zorder=5)
h.append(Line2D([0], [0], marker="o", color="none", markerfacecolor="#c98d3a", markeredgecolor="white",
                markersize=8, label=f"Памятники регионального значения ({len(reg)})"))
fed.to_crs(CRS_M).plot(ax=ax, color="#8c2d2d", markersize=42, edgecolor="white", linewidth=0.8, zorder=6)
h.append(Line2D([0], [0], marker="o", color="none", markerfacecolor="#8c2d2d", markeredgecolor="white",
                markersize=10, label=f"Памятники федерального значения ({len(fed)})"))

ny = okn_d[okn_d.name.str.contains("Ниеншанц", na=False)]
if len(ny):
    p = ny.to_crs(CRS_M).geometry.iloc[0]
    ax.annotate("Крепость Ниеншанц (раскоп «Охта-1»)", xy=(p.x, p.y), xytext=(p.x - 900, p.y + 500), fontsize=9,
                fontweight="bold", color="#8c2d2d", path_effects=maptools.HALO,
                arrowprops=dict(arrowstyle="->", color="#8c2d2d"), zorder=7)

gpd.GeoSeries([LAND_V2], crs=CRS_M).boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.6, zorder=4)
h.append(Line2D([0], [0], color="#1f4e79", lw=1.6, label="Граница МО Малая Охта"))
maptools.finish(fig, ax, EXTENT, h, title="Наследие и зелень", crs=CRS_M, basemap_style="gray", basemap_alpha=0.95)
maptools.save_map(fig, OUT / "03_heritage.png")
print("3 готова")


3 готова


In [8]:
# Карта 2: возраст жилых домов (HOITH)
PERIODS = [("до 1917", 1700, 1917, "#8c2d2d"), ("1917–1945", 1917, 1946, "#c98d3a"),
           ("1946–1960", 1946, 1961, "#7fb069"), ("1961–1980", 1961, 1981, "#4a7aa5"),
           ("1981+", 1981, 2027, "#6c5b7b")]

fig, ax = maptools.map_figure(full=True)
h = []
for name, y0, y1, color in PERIODS:
    sel = house_age[(house_age.year >= y0) & (house_age.year < y1)]
    if not len(sel):
        continue
    sel.to_crs(CRS_M).plot(ax=ax, facecolor=color, edgecolor="white", linewidth=0.3, zorder=3)
    h.append(Patch(facecolor=color, edgecolor="white", label=f"Построено в {name} ({len(sel)} домов)"))
draw_water(ax)
h.append(Patch(facecolor=WATER_COLOR, alpha=0.9, label="Вода"))
gpd.GeoSeries([LAND_V2], crs=CRS_M).boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.8, zorder=5)
h.append(Line2D([0], [0], color="#1f4e79", lw=1.8, label="Граница МО Малая Охта"))

maptools.finish(fig, ax, EXTENT, h, title="Возраст жилых домов", crs=CRS_M, basemap_style="gray", basemap_alpha=0.9)
maptools.save_map(fig, OUT / "02_house_age.png")
print("2 готова; домов с годом:", int(house_age.year.notna().sum()))


2 готова; домов с годом: 1095


In [9]:
# График населения: ряд 2010–2025 по реальным годам (неравные интервалы — числовая ось)
import matplotlib.pyplot as plt

pop = pd.read_csv(PROJECT / "data/raw/population_spb_mo_2010_2025.csv", encoding="utf-8-sig")
COLS = {"ВПН-2010": 2010.25, "01.01.2018": 2018.0, "01.01.2019": 2019.0, "01.01.2020": 2020.0,
        "01.01.2021": 2021.0, "01.01.2022": 2022.0, "ВПН-2020 (01.10.2021)": 2021.75,
        "01.01.2023": 2023.0, "01.01.2024": 2024.0, "01.01.2025": 2025.0}

fig, ax = plt.subplots(figsize=(10, 6))
for name, color in [("МО Малая Охта", "#B33A2B"), ("МО Большая Охта", "#1F4E79")]:
    row = pop[pop["территория"] == name].iloc[0]
    xs = [v for c, v in COLS.items() if c in pop.columns and pd.notna(row[c])]
    ys = [int(row[c]) for c, v in COLS.items() if c in pop.columns and pd.notna(row[c])]
    census = [i for i, (c, v) in enumerate(zip([c for c, v in COLS.items() if c in pop.columns and pd.notna(row[c])], xs))
              if "ВПН" in str(c)]
    ax.plot(xs, ys, color=color, lw=2.8, marker="o", markersize=6, label=name, zorder=3)
    for i in census:
        ax.plot(xs[i], ys[i], marker="o", markersize=10, mfc="none", mec=color, mew=2, zorder=4)

ax.set_xlim(2009.5, 2025.5); ax.set_ylim(40000, 62000)
ax.set_xticks([2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024])
ax.grid(axis="y", color="#E2E8F0", lw=0.7)
ax.set_axisbelow(True)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#B5C2CD")
ax.tick_params(colors="#52514e", labelsize=11)
ax.set_ylabel("человек", fontsize=11, color="#52514e")
ax.annotate("переписи (крупные маркеры)", xy=(0.02, 0.04), xycoords="axes fraction",
            fontsize=10, color="#52514e")
ax.legend(loc="lower left", bbox_to_anchor=(0.02, 0.10), frameon=False, fontsize=12)
fig.savefig(OUT / "08_population.png", dpi=200, facecolor="white")
plt.close(fig)
print("график населения готов:", (OUT / "08_population.png").name)


график населения готов: 08_population.png


In [10]:
# Фоновые карты для титула и финала: те же слои без легенды, севера, линейки и рамки
# (в фон не должны попадать обрезанные легенды и служебные элементы)
def bg_map(path, draw):
    fig, ax = maptools.map_figure(full=True)
    maptools.set_extent(ax, *EXTENT)
    keep = (ax.get_xlim(), ax.get_ylim())
    maptools.basemap(ax, "gray", CRS_M, 0.9)
    ax.set_xlim(*keep[0]); ax.set_ylim(*keep[1])
    draw(ax)
    ax.set_axis_off()
    maptools.no_yo(fig)
    fig.savefig(OUT / path, dpi=maptools.DPI)
    plt.close(fig)
    print(path, "готов")

def draw_bg1(ax):
    water.to_crs(CRS_M).plot(ax=ax, facecolor="#aec6d8", edgecolor="none", alpha=0.85, zorder=1)
    gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M).boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=2.4, zorder=4)
    gpd.GeoSeries([V2], crs=4326).to_crs(CRS_M).boundary.plot(ax=ax, edgecolor="#8c2d2d", linewidth=2.6, zorder=5)
    gpd.GeoSeries([V1], crs=4326).to_crs(CRS_M).plot(ax=ax, facecolor="#e9b96e", edgecolor="none", alpha=0.55, zorder=3)

def draw_bg18(ax):
    draw_water(ax)
    green[green.kind.isin(["park", "forest", "wood"])].to_crs(CRS_M).plot(
        ax=ax, facecolor="#a9c79a", edgecolor="none", alpha=0.6, zorder=2)
    okn[okn.cat == "Федерального значения"].to_crs(CRS_M).plot(
        ax=ax, color="#8c2d2d", markersize=52, edgecolor="white", linewidth=0.8, zorder=6)
    okn[okn.cat == "Регионального значения"].to_crs(CRS_M).plot(
        ax=ax, color="#c98d3a", markersize=34, edgecolor="white", linewidth=0.6, zorder=5)

bg_map("01_bg_clean.png", draw_bg1)
bg_map("03_bg_clean.png", draw_bg18)


01_bg_clean.png готов


03_bg_clean.png готов
